In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA

from imblearn.over_sampling import SMOTE
import joblib

In [18]:
df = pd.read_csv("diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [19]:
df.info()
print("---------------------------")
print(df.describe())
print("---------------------------")
df["Outcome"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
---------------------------
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458

,count
Outcome,
0,500
1,268


# Feature Engineering


In [20]:

# Glucose × BMI interaction
df["Glucose_BMI"] = df["Glucose"] * df["BMI"]

# BMI × Age interaction
df["BMI_Age"] = df["BMI"] * df["Age"]

# Age Groups
df["Age_group"] = pd.cut(
    df["Age"],
    bins=[18, 30, 50, 100],
    labels=["young", "mid_age", "senior"]
)

# Insulin to Glucose Ratio
df["Insulin_Glucose"] = df["Insulin"] / (df["Glucose"] + 1)

# Convert categorical features
df = pd.get_dummies(df, drop_first=True)


# Split Features and Target


In [21]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [22]:
# Handle Imbalanced Data

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [23]:
# Feature Scaling

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [24]:

# PCA

pca = PCA(n_components=5)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)

print("Total Variance Retained:")
print(sum(pca.explained_variance_ratio_))


Explained Variance Ratio:
[0.28205916 0.19443044 0.11252835 0.09287808 0.08080346]
Total Variance Retained:
0.762699488997596


In [25]:

# Train Model

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [26]:
# Prediction

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [27]:
# Evaluation

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.7662337662337663
ROC-AUC: 0.8357407407407406

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.76      0.81       100
           1       0.64      0.78      0.70        54

    accuracy                           0.77       154
   macro avg       0.75      0.77      0.75       154
weighted avg       0.78      0.77      0.77       154



In [28]:
# Save Model

joblib.dump(model, "diabetes_model.pkl")

print("\nModel saved successfully!")


Model saved successfully!
